# ランダムフォレストと OOB スコア

**出典**　機械学習 4 分野の教科書 — 第 4 章 1-C アンサンブル・評価・前処理 ／ 02 ランダムフォレスト：もう一段のランダム性

**このノートのパス**　`ml-code/04_ensemble_eval/01_random_forest.ipynb`

**このノートで確かめること**
**木の本数を増やしても過学習しない**こと、**OOB スコアで別途の検証データなしに評価できる**ことを確かめる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `scikit-learn, pandas` |
| 実行時間の目安 | 約 20 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [1]:
# 未導入なら実行する（Colab など）
# !pip install -q scikit-learn pandas

In [2]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

data = load_breast_cancer()                 # sklearn 同梱。オフラインで動く
feature_names = data.feature_names
X_tr, X_te, y_tr, y_te = train_test_split(
    data.data, data.target, test_size=0.2, stratify=data.target, random_state=0)

rf = RandomForestClassifier(
    n_estimators=500,      # 木の本数。多いほど良いが、増やしても過学習はしない（頭打ちになるだけ）
    max_features="sqrt",   # 各分割で使う特徴量の数。工夫②の核心
    min_samples_leaf=1,    # 葉の最小サンプル数。バギングでは木を深く育てるので既定のままでよい
    oob_score=True,        # 学習に使われなかったデータで自動検証
    n_jobs=-1,             # 並列実行（バギングは独立なので並列化できる）
    random_state=0,
).fit(X_tr, y_tr)

print("OOB スコア:", rf.oob_score_)

# 特徴量重要度（不純度の減少量ベース）
imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
print(imp.head(10))

OOB スコア: 0.9626373626373627
worst concave points    0.133040
worst perimeter         0.132673
worst area              0.113934
worst radius            0.111771
mean concave points     0.101282
mean concavity          0.056874
mean perimeter          0.050433
mean area               0.041944
mean radius             0.037770
worst concavity         0.035327
dtype: float64


### 本数を増やしても過学習しないことを、数字で確かめる

In [3]:
from sklearn.metrics import roc_auc_score

for n in [50, 100, 500, 1500]:
    m = RandomForestClassifier(n_estimators=n, max_features="sqrt",
                               oob_score=True, n_jobs=-1, random_state=0).fit(X_tr, y_tr)
    print(f"n_estimators={n:>4}  OOB={m.oob_score_:.4f}  "
          f"test AUC={roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]):.4f}")

n_estimators=  50  OOB=0.9582  test AUC=0.9706
n_estimators= 100  OOB=0.9626  test AUC=0.9729


n_estimators= 500  OOB=0.9626  test AUC=0.9714


n_estimators=1500  OOB=0.9626  test AUC=0.9696


### `feature_importances_` の偏りを確かめる — permutation importance と比べる

少ない値しか取らない特徴量（プラン・地域・会員か・問い合わせ回数）で目的が決まるデータに、
**目的と無関係な `customer_id`**（全行で値が異なる）を 1 列足す。
不純度ベースの重要度は、値の種類が多い列ほど分割点を多く見つけられるので、無関係な ID を上位に押し上げる。
permutation importance は「テストデータでその列だけを並べ替えたら、スコアがどれだけ落ちるか」を測るので、無関係な列はほぼ 0 になる。

In [4]:
from sklearn.inspection import permutation_importance
import numpy as np

# 少ない値しか取らない特徴量 4 つで目的が決まるデータに、目的と無関係な customer_id（全行で値が異なる）を足す
rng = np.random.default_rng(0)
n = 1000
Xc = pd.DataFrame({
    "plan":       rng.integers(0, 3, n),   # 契約プラン（3 種類）
    "region":     rng.integers(0, 4, n),   # 地域（4 種類）
    "is_member":  rng.integers(0, 2, n),   # 会員か（2 値）
    "n_contacts": rng.integers(0, 5, n),   # 問い合わせ回数（0〜4）
})
logit = 1.2 * (Xc.plan - 1) + 0.8 * Xc.is_member - 0.5 * (Xc.n_contacts - 2) + 0.3 * (Xc.region == 2)
yc = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
Xc["customer_id"] = rng.permutation(n)     # 目的とは無関係

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, stratify=yc, random_state=0)
rf_c = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0).fit(Xc_tr, yc_tr)

result = permutation_importance(rf_c, Xc_te, yc_te, scoring="roc_auc",
                                n_repeats=20, random_state=0)   # テストデータで列を 1 本ずつ並べ替える
print(pd.DataFrame({
    "不純度ベース": rf_c.feature_importances_,
    "permutation": result.importances_mean,
}, index=Xc.columns).round(4).sort_values("不純度ベース", ascending=False))

             不純度ベース  permutation
customer_id  0.5169      -0.0251
plan         0.1995       0.1023
n_contacts   0.1433       0.0277
region       0.0976      -0.0153
is_member    0.0427       0.0151


## 読み取りどころ

- **本数を増やしてもテスト性能は劣化しない**（頭打ちになるだけ）。これは各木が独立に学習され、平均を取っているためで、ブースティングとは性質が違う。
- **OOB スコアは、別に検証データを切り出さなくても得られる推定値**である。ブートストラップ標本に選ばれなかったデータ（約 37%）で評価している。

⚠️ **`feature_importances_`（不純度ベース）は、カーディナリティの高い特徴を過大評価する。**
連続値や ID のような列があると不当に上位に来る。信頼できる重要度が要るなら `permutation_importance` か SHAP を使うこと。上の比較では、無関係な `customer_id` が不純度ベースで 1 位になり、permutation では 0 以下になった。